# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 03** </center>
---
**Profesor**: Pablo Camarillo Ramirez

In [29]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Lab 03", "spark://spark-master:7077")
su._spark

In [30]:
!pwd

/opt/spark/work-dir


In [31]:
!ls /opt/spark/work-dir/data/qcommerce/

qcommerce.csv


In [32]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns_types = [("Order_ID", "long"),
                 ("Company", "string"),
                 ("City", "string"),
                 ("Customer_Age", "int"),
                 ("Order_Value", "float"),
                 ("Delivery_Time_Min", "float"),
                 ("Distance_Km", "float"),
                 ("Items_Count", "float"),
                 ("Product_Category", "string"),
                 ("Payment_Method", "string"),
                 ("Customer_Rating", "float"),
                 ("Discount_Applied", "float"),
                 ("Delivery_Partner_Rating", "float")
                 ]

qcommerce_schema = SparkUtils.generate_schema(columns_types)
print(qcommerce_schema.__class__)

qcommerce_df = su._spark.read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/")

qcommerce_df.printSchema()
print(f"número de registros: {qcommerce_df.count()}")
qcommerce_df.show(5)


<class 'pyspark.sql.types.StructType'>
root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



número de registros: 1000000
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3

In [33]:
from pyspark.sql.functions import when, count, isnull

qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [34]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [35]:
qcommerce_null_count_df_2 = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df_2.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



## Cleaning Dataset

In [37]:
qcommerce_df_wo_nulls_dropna = qcommerce_df.dropna()
print(f"número de registros despues de usar el metodo 'dropna()':{qcommerce_df_wo_nulls_dropna.count()}")

número de registros despues de usar el metodo 'dropna()':780992


In [39]:
qcommerce_clean_df = qcommerce_df.fillna({
    'City': 'Uknown',
    'Items_Count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})
print(f"número de registros despues de usar el metodo 'fillna()':{qcommerce_clean_df.count()}")

número de registros despues de usar el metodo 'fillna()':1000000


In [44]:
SparkUtils.count_nulls(qcommerce_clean_df).show()

+--------+-------+----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+------------+
|Order_ID|Company|City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Country_Code|
+--------+-------+----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+------------+
|       0|      0|   0|           0|          0|                0|          0|          0|               0|             0|              0|               0|                      0|           0|
+--------+-------+----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+------------+



## Create new columns

In [40]:
from pyspark.sql.functions import lit
qcommerce_clean_df = qcommerce_clean_df.withColumn("Country_Code", lit("IN"))
qcommerce_clean_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Country_Code|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

# Task 1
    Create Delivery_SLA with withColumn + when:
    Delivery_Time_Min ≤ 15 → "FAST", 15--20 → "ON_TIME", > 20 →
    "LATE"
    filter only Delivery_SLA = "LATE" and orderBy Delivery_Time_Min (desc).
    Display: Order_ID, Company, City, Delivery_Time_Min, Delivery_SLA.

In [ ]:
from pyspark.sql.functions import col
qcommerce_task1_df = qcommerce_clean_df.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST") \
                                                    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON_TIME")\
                                                    .when(col("Delivery_Time_Min") > 20, "LATE")) \
                                                    .filter(col("Delivery_SLA") == "LATE") \
                                                    .orderBy(col("Delivery_Time_Min"), ascending=False) \
                                                    .select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA")
qcommerce_task1_df.show()

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

# Task 2
    Create Effective_Order_Value = Order_Value * (1 - Discount_Applied).
    Create Price_Bucket with when:
    < 200 → "LOW", 200--500 → "MEDIUM", > 500 → "HIGH"
    groupBy(Price_Bucket) and compute count(*) and avg(Effective_Order_Value).
    orderBy average effective value (desc). 

In [46]:
from pyspark.sql.functions import avg, count
qcommerce_task2_df = qcommerce_clean_df.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))) \
            .withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW") \
                                        .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM").otherwise("HIGH")) \
            .groupBy("Price_Bucket").agg(
                count("*").alias("Count_Price_Bucket"),
                avg("Effective_Order_Value").alias("Avg_Effective_Value")
            ).orderBy("Avg_Effective_Value", ascending=False)
qcommerce_task2_df.show()

+------------+------------------+-------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Value|
+------------+------------------+-------------------+
|        HIGH|            268953|  740.4337238893675|
|      MEDIUM|            210429|  358.0973233400432|
|         LOW|            520618| 21.591204157544553|
+------------+------------------+-------------------+



# Task 3

    Create Age_Group with withColumn + when:
    < 25 → "YOUNG", 25--44 → "ADULT", ≥ 45 → "SENIOR"
    filter invalid ages (nulls, < 0, or > 100).
    groupBy(Age_Group, Product_Category) and compute:
    orders = count(*), avg_order_value = avg(Order_Value)
    orderBy(Age_Group asc, orders desc) to find top categories per segment.

In [51]:
qcommerce_task3_df = qcommerce_clean_df.withColumn("Age_Group", when(col("Customer_Age") < 25, "YOUNG") \
                                                .when((col("Customer_Age") >= 25) & (col("Customer_Age") < 44), "ADULT") \
                                                .otherwise("SENIOR")) \
                                                .filter((col("Customer_Age") > 0) & (col("Customer_Age") <= 100)) \
                                                .groupBy(col("Age_Group"), col("Product_Category")).agg(
                                                    count("*").alias("count_age_group"),
                                                    avg("Order_Value").alias("avg_order_value")
                                                ).orderBy(col("Age_Group"), ascending=False)

qcommerce_task3_df.show()

+---------+-------------------+---------------+-----------------+
|Age_Group|   Product_Category|count_age_group|  avg_order_value|
+---------+-------------------+---------------+-----------------+
|    YOUNG|      Personal Care|          23687| 568.364765176416|
|    YOUNG|             Snacks|          23774|571.4999866301071|
|    YOUNG|              Dairy|          24235|572.3825174487889|
|    YOUNG|          Beverages|          23885|571.5616625072918|
|    YOUNG|          Household|          23942|577.2101138994992|
|    YOUNG|          Groceries|          23881|571.6804158469649|
|    YOUNG|Fruits & Vegetables|          23660|570.9057398116498|
|   SENIOR|      Personal Care|          54025|570.9699125884629|
|   SENIOR|             Snacks|          54339|572.3606182995497|
|   SENIOR|          Groceries|          54353|571.9842917366681|
|   SENIOR|          Household|          54235|571.3643088361069|
|   SENIOR|Fruits & Vegetables|          53988|573.6265609516801|
|   SENIOR

In [52]:
su._spark.stop()